# Task 4 - Graph topology ingestion into Neo4j

Kafka Connect consumes only node and edge topics. Cypher handlers
use `MERGE`, create placeholder endpoints when necessary, and reconcile delete
events without Spark.

## Direct graph ingestion

The Neo4j sink subscribes only to `cpg.nodes.v1` and `cpg.edges.v1`; metadata
never enters this branch. That boundary is both an assignment requirement and a
useful design constraint, because it prevents Spark from becoming a graph relay.
Stable IDs are used in Cypher `MERGE` operations, explicit delete events remove
obsolete topology, and a uniqueness constraint provides a second guard against
duplicate nodes.

Node and edge topics do not share a global arrival order. Rather than delaying
all edges until a separate coordination step proves every endpoint exists, the
edge handler creates placeholder `CPGNode` endpoints. A later node event merges
properties into the same ID. This makes retries and cross-topic ordering
converge without an external buffer.

We kept one generic node label and relationship type, with AST/CFG/DFG/CALL
kinds stored as properties. Dynamic labels would produce prettier Browser
queries, but they complicate parameterized connector Cypher and schema changes.
Failed records go to a dedicated DLQ so that connector tolerance cannot silently
hide ingestion errors.

In [1]:
import json
import subprocess
from pathlib import Path

root = Path('..').resolve()
result = subprocess.run(
    ['docker', 'compose', 'exec', '-T', 'connect', 'curl', '-fsS',
     'http://localhost:8083/connectors/cpg-neo4j-sink/status'],
    cwd=root, capture_output=True, text=True, check=True,
)
status = json.loads(result.stdout)
print(json.dumps(status, indent=2))
assert status['connector']['state'] == 'RUNNING'
assert status['tasks'] and all(task['state'] == 'RUNNING' for task in status['tasks'])
print('PASS: connector and task are RUNNING')

{
  "name": "cpg-neo4j-sink",
  "connector": {
    "state": "RUNNING",
    "worker_id": "connect:8083"
  },
  "tasks": [
    {
      "id": 0,
      "state": "RUNNING",
      "worker_id": "connect:8083"
    }
  ],
  "type": "sink"
}
PASS: connector and task are RUNNING


In [2]:
import json
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root / 'scripts'))
from capture_replay_evidence import neo4j_snapshot

evidence = json.loads((root / 'evidence/runtime/verification.json').read_text(encoding='utf-8'))
file_id = evidence['replay_file']['file_id']
snapshot = neo4j_snapshot(file_id)
print(json.dumps(snapshot, indent=2))
assert snapshot['nodes'] == snapshot['unique_nodes']
assert snapshot['edges'] == snapshot['unique_edges']
assert {'AST', 'CFG', 'DFG', 'CALL'} <= set(snapshot['edge_kinds'])
print('PASS: Neo4j IDs are unique and all graph edge kinds exist')

{
  "nodes": 62550,
  "unique_nodes": 62550,
  "edges": 77873,
  "unique_edges": 77873,
  "file_nodes": 29,
  "file_edges": 36,
  "edge_kinds": {
    "AST": 57960,
    "CALL": 2593,
    "CFG": 7248,
    "DFG": 10072
  }
}
PASS: Neo4j IDs are unique and all graph edge kinds exist


In [3]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
result = subprocess.run(
    ['docker', 'compose', 'exec', '-T', 'broker', 'kafka-get-offsets',
     '--bootstrap-server', 'broker:29092', '--topic', 'cpg.neo4j-dlq.v1'],
    cwd=root, capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
offset = sum(int(line.rsplit(':', 1)[1]) for line in result.stdout.splitlines() if ':' in line)
assert offset == 0
print('PASS: Neo4j connector DLQ is empty')

cpg.neo4j-dlq.v1:0:0
PASS: Neo4j connector DLQ is empty


![Neo4j Browser showing CPG nodes and relationships](figures/neo4j-browser.png)

*Neo4j Browser showing CPG nodes and relationships*

## Reflection

The connector and its task remained `RUNNING`, total counts
matched distinct IDs, every graph edge kind was present, and the DLQ stayed
empty. The Neo4j Browser image provides a separate visual check of the resulting
topology. The main failure occurred during registration: the REST request could
return before the asynchronous task reached a runnable state. Registration is
now idempotent and followed by bounded polling, which distinguishes “request
accepted” from “sink ready” and makes repeated stack startup reliable.